# Notebook 2: How Do You Know If It Worked?

**Plan A — Measuring Progress**

In Notebook 1 you built an encoded magic state and verified it in an ideal simulator. But real quantum hardware is noisy. This notebook answers: *How do we quantify how good our preparation is, and how do we turn those measurements into a single score the ratchet can optimize?*

**What you will learn:**
1. What noise does to your encoded state
2. How to measure logical operators and compute the magic witness
3. The scoring formula: quality, acceptance, and cost
4. How to identify dominant failure modes
5. How parameter choices affect the score

In [ ]:
%matplotlib inline
import sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from math import pi, sqrt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

from autoresearch_quantum.codes.four_two_two import (
    build_preparation_circuit, encoded_magic_statevector,
    STABILIZERS, MEASUREMENT_OPERATORS, DATA_QUBITS,
)
from autoresearch_quantum.experiments.encoded_magic_state import build_circuit_bundle
from autoresearch_quantum.models import (
    ExperimentSpec, RungConfig, EvaluationMetrics,
    QualityWeights, CostWeights, ScoreConfig, SearchSpaceConfig,
    TierPolicyConfig, HardwareConfig,
)
from autoresearch_quantum.execution.local import LocalCheapExecutor
from autoresearch_quantum.execution.analysis import (
    logical_magic_witness, stability_score, summarize_context,
    local_memory_records,
)
from autoresearch_quantum.execution.backends import resolve_backend
from autoresearch_quantum.execution.transpile import (
    transpile_circuits, count_two_qubit_gates, circuit_metadata, runtime_estimate,
)
from autoresearch_quantum.scoring.score import (
    score_metrics, weighted_acceptance_cost, factory_throughput_score,
)

print("All imports successful.")

In [ ]:
from autoresearch_quantum.teaching import LearningTracker
from autoresearch_quantum.teaching.assess import quiz, predict_choice, reflect, order, checkpoint_summary
tracker = LearningTracker("plan_a_02")
print("Learning tracker active.")

---
## 1. Recap: The Ideal Encoded Magic State

Let us quickly rebuild the preparation circuit and its ideal statevector from Notebook 1.

In [ ]:
prep = build_preparation_circuit("h_p", "cx_chain")
ideal_state = Statevector.from_instruction(prep)

# Verify stabilizers
for name, stab in STABILIZERS.items():
    exp_val = ideal_state.expectation_value(stab)
    print(f"{name}: <{name}> = {exp_val.real:+.6f}")

print(f"\nIdeal self-fidelity: {state_fidelity(ideal_state, ideal_state):.6f}")
prep.draw('mpl', style='iqp')

### Recap check

Before we add noise, make sure the Notebook 1 concepts are solid.

In [ ]:
quiz(tracker, "q1_stabilizer_eigenvalue",
    question="What do the stabilizer eigenvalues tell us about a quantum state?",
    options=[
        "They measure the energy of the system",
        "Eigenvalue +1 means the state is in the codespace (no error detected)",
        "They tell us which logical qubit is |0\u27E9 vs |1\u27E9",
        "They count the number of entangled qubits",
    ],
    correct=1,
    section="1. Recap",
    bloom="remember",
    explanation=(
        "Stabilizer eigenvalue +1 means the state satisfies the code constraints. "
        "If any single-qubit error occurs, at least one stabilizer flips to \u22121."
    ))

---
## 2. Adding Noise: The fake_brisbane Backend

Real hardware introduces gate errors, readout errors, and decoherence. Qiskit provides **fake backends** that model the noise of real IBM devices without requiring API access.

In [ ]:
# Load the fake_brisbane backend and extract its noise model
backend = resolve_backend("fake_brisbane")
noise_model = NoiseModel.from_backend(backend)

print(f"Backend: {backend.name}")
print(f"Qubits:  {backend.num_qubits}")
print(f"Basis gates: {noise_model.basis_gates}")
print(f"Noise instructions: {len(noise_model.to_dict()['errors'])} error channels")

### What does noise do?

The fake_brisbane backend simulates realistic noise from IBM's 127-qubit Eagle processor. Gate errors, readout errors, and crosstalk all contribute.

In [ ]:
predict_choice(tracker, "q2_noise_effect",
    question="When we run the same circuit with noise, what happens to the syndrome distribution?",
    options=[
        "Syndrome is still always '00' \u2014 noise is too small to matter",
        "Some shots will have non-zero syndrome \u2014 noise causes detectable errors",
        "All shots will have non-zero syndrome \u2014 noise is overwhelming",
    ],
    correct=1,
    section="2. Adding noise",
    bloom="understand",
    explanation=(
        "Realistic noise causes some fraction of shots to leave the codespace, "
        "producing non-zero syndromes. Typical acceptance rates are 40\u201380% for current hardware."
    ))

### Ideal vs Noisy Simulation

Let us run the same preparation circuit twice: once ideal, once with the Brisbane noise model.

In [ ]:
spec = ExperimentSpec(
    rung=1, seed_style="h_p", encoder_style="cx_chain",
    verification="both", postselection="all_measured",
    target_backend="fake_brisbane", noise_backend="fake_brisbane",
    optimization_level=2, shots=512, repeats=1,
)
bundle = build_circuit_bundle(spec)

# Transpile the acceptance circuit
transpiled = transpile_circuits([bundle.acceptance], spec, backend)[0]

# Ideal simulation
ideal_sim = AerSimulator()
ideal_result = ideal_sim.run(transpiled, shots=512, memory=True, seed_simulator=42).result()
ideal_counts = ideal_result.get_counts()

# Noisy simulation
noisy_sim = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    coupling_map=backend.coupling_map,
)
noisy_result = noisy_sim.run(transpiled, shots=512, memory=True, seed_simulator=42).result()
noisy_counts = noisy_result.get_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
plot_histogram(ideal_counts, ax=ax1, title="Ideal Simulation")
plot_histogram(noisy_counts, ax=ax2, title="Noisy Simulation (fake_brisbane)")
plt.tight_layout()
plt.show()

> **Observe:** The ideal histogram shows only a few bitstrings (those consistent with the codespace). The noisy histogram is spread across many more outcomes — noise is kicking the state out of the codespace.

---
## 3. Postselection: Filtering Bad Shots

Each bitstring has two parts: **syndrome bits** (from stabilizer checks) and **data bits** (from qubit measurements). Postselection keeps only shots where the syndrome indicates "no error detected."

In [ ]:
# Parse raw shots into records
memory = noisy_result.get_memory(transpiled)
records = local_memory_records(memory, [creg.name for creg in transpiled.cregs])

total = len(records)
accepted = sum(
    1 for r in records
    if all(b == '0' for b in r.get('syndrome', ''))
)
rejected = total - accepted

print(f"Total shots:    {total}")
print(f"Accepted (syndrome=00): {accepted}  ({accepted/total:.1%})")
print(f"Rejected:       {rejected}  ({rejected/total:.1%})")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Accepted", "Rejected"], [accepted, rejected], color=["#2ecc71", "#e74c3c"])
ax.set_ylabel("Shots")
ax.set_title("Postselection Results")
plt.tight_layout()
plt.show()

In [ ]:
quiz(tracker, "q3_acceptance_cost",
    question="If the acceptance rate is 50%, what does that mean for the experiment?",
    options=[
        "Half the qubits failed",
        "Half the shots were discarded \u2014 we need 2x shots for the same statistics",
        "The circuit has 50% fidelity",
        "The code corrected half the errors",
    ],
    correct=1,
    section="3. Postselection",
    bloom="understand",
    explanation=(
        "Acceptance rate = fraction of shots surviving postselection. "
        "At 50%, you need twice as many total shots. This is a direct cost."
    ))
checkpoint_summary(tracker, "3. Postselection")

> **Key Insight:** The acceptance rate is the fraction of shots that survive postselection. Under noise it can drop significantly — this is the code *working* by detecting errors. But a low acceptance rate means you need more shots to get the same statistical precision, which increases cost.

---
## 4. Measuring Logical Operators

To certify that we have an encoded *magic* state (not just any codespace state), we need to measure three logical operators:

| Operator | Pauli String | Physical Meaning |
|----------|-------------|------------------|
| Logical X |  X_2$ | X-component of logical qubit 0 |
| Logical Y |  Z_1 X_2$ | Y-component of logical qubit 0 |
| Spectator Z |  Z_2$ | Z-component of logical qubit 1 (should be +1 if undisturbed) |

Each requires a separate circuit with basis rotations before measurement.

In [ ]:
# Run all witness circuits on the noisy simulator
context_names = ["acceptance", "logical_x", "logical_y", "spectator_z"]
raw_circuits = [bundle.acceptance] + [bundle.witness_circuits[k] for k in ["logical_x", "logical_y", "spectator_z"]]
transpiled_circuits = transpile_circuits(raw_circuits, spec, backend)

results = {}
for name, circuit in zip(context_names, transpiled_circuits):
    result = noisy_sim.run(circuit, shots=512, memory=True, seed_simulator=42).result()
    mem = result.get_memory(circuit)
    recs = local_memory_records(mem, [creg.name for creg in circuit.cregs])
    summary = summarize_context(
        recs,
        syndrome_labels=list(circuit.metadata.get('syndrome_labels', [])),
        postselection=str(circuit.metadata.get('postselection', 'none')),
        operator=MEASUREMENT_OPERATORS.get(name),
    )
    results[name] = summary
    print(f"{name:15s}  acceptance_rate={summary['acceptance_rate']:.3f}  expectation={summary['expectation']:+.4f}")

In [ ]:
quiz(tracker, "q4_three_circuits",
    question="Why does the experiment use 3 separate circuits instead of measuring all operators at once?",
    options=[
        "The operators don't commute, so measuring one disturbs the others",
        "It's a software limitation",
        "Each operator requires different ancilla qubits",
    ],
    correct=0,
    section="4. Logical operators",
    bloom="analyze",
    explanation=(
        "Logical X and Logical Y do not commute. Measuring one collapses the state "
        "into an eigenstate that invalidates the other measurement."
    ))

---
## 5. The Magic Witness Formula

The **logical magic witness** combines the three operator expectations into a single number between 0 and 1:

9951W = rac{1 + rac{\langle X_L angle + \langle Y_L angle}{\sqrt{2}}}{2} 	imes rac{1 + \langle Z_{	ext{spectator}} angle}{2}9951

- The first factor checks the "magic" — a perfect T-state has $\langle X_L angle = \langle Y_L angle = 1/\sqrt{2}$, giving 1.0.
- The second factor checks spectator integrity — the second logical qubit should be in $|0_Langle$, giving $\langle Z_{	ext{spectator}} angle = +1$.
- For a perfect encoded T-state:  = 1.0$.
- For a maximally mixed state:  pprox 0.25$.

In [ ]:
lx = results["logical_x"]["expectation"]
ly = results["logical_y"]["expectation"]
sz = results["spectator_z"]["expectation"]

magic_factor = (1.0 + (lx + ly) / sqrt(2.0)) / 2.0
spectator_factor = (1.0 + sz) / 2.0
witness = magic_factor * spectator_factor

print(f"Logical X expectation:    {lx:+.4f}")
print(f"Logical Y expectation:    {ly:+.4f}")
print(f"Spectator Z expectation:  {sz:+.4f}")
print(f"")
print(f"Magic factor:     {magic_factor:.4f}")
print(f"Spectator factor: {spectator_factor:.4f}")
print(f"Witness W:        {witness:.4f}")

# Verify against the library function
lib_witness = logical_magic_witness(lx, ly, sz)
print(f"Library witness:  {lib_witness:.4f}  (should match)")

### The magic witness formula

$$W = \frac{1 + (\langle X_L \rangle + \langle Y_L \rangle)/\sqrt{2}}{2} \times \frac{1 + \langle Z_{\text{spectator}} \rangle}{2}$$

For a perfect T-state: $\langle X_L \rangle = \langle Y_L \rangle = 1/\sqrt{2}$ and $\langle Z_{\text{spec}} \rangle = 1$, giving $W = 1.0$.

In [ ]:
quiz(tracker, "q5_ideal_witness",
    question="For a perfect (noiseless) T-state, what is the magic witness value?",
    options=["0.0", "0.5", "1/\u221A2 \u2248 0.707", "1.0"],
    correct=3,
    section="5. Witness formula",
    bloom="apply",
    explanation=(
        "magic_factor = (1 + (1/\u221A2 + 1/\u221A2)/\u221A2) / 2 = (1+1)/2 = 1. "
        "spectator_factor = (1+1)/2 = 1. Product = 1.0."
    ))
checkpoint_summary(tracker, "5. Witness formula")

---
## 6. Noisy Fidelity via Density Matrix

An alternative quality metric is the **state fidelity** — how close the noisy output is to the ideal encoded T-state.

In [ ]:
# Noisy fidelity via density matrix on the LOGICAL (untranspiled) 4-qubit circuit.
# Note: density matrix simulation on the transpiled 127-qubit circuit would require
# 2^127 x 2^127 matrix entries — impossible on any current machine!
# Instead, we apply the noise model to the compact 4-qubit prep circuit.

target = encoded_magic_statevector()

# Build a small noise model for just the gates we use
from qiskit_aer.noise import NoiseModel, depolarizing_error
simple_noise = NoiseModel()
simple_noise.add_all_qubit_quantum_error(depolarizing_error(0.001, 1), ['h', 'p', 'rz', 'ry', 'sx', 'x', 'u'])
simple_noise.add_all_qubit_quantum_error(depolarizing_error(0.01, 2), ['cx'])

dm_circuit = build_preparation_circuit("h_p", "cx_chain")
dm_circuit.save_density_matrix()

density_sim = AerSimulator(method="density_matrix", noise_model=simple_noise)
dm_result = density_sim.run(dm_circuit).result()
noisy_dm = dm_result.data(0)["density_matrix"]
noisy_fidelity = state_fidelity(noisy_dm, target)

ideal_fidelity = state_fidelity(Statevector.from_instruction(bundle.prep), target)

print(f"Ideal fidelity (circuit vs target):  {ideal_fidelity:.6f}")
print(f"Noisy fidelity (density matrix):     {noisy_fidelity:.6f}")
print(f"Fidelity loss from noise:            {ideal_fidelity - noisy_fidelity:.6f}")
print(f"\nNote: This uses a simplified depolarizing noise model on the 4-qubit")
print(f"logical circuit. The full LocalCheapExecutor uses the backend-specific")
print(f"noise model on the transpiled circuit via shot-based simulation.")

In [ ]:
quiz(tracker, "q6_witness_vs_fidelity",
    question="The witness and fidelity both measure quality. How do they differ?",
    options=[
        "They are the same thing",
        "Fidelity measures overlap with the ideal state; the witness tests magic-state properties specifically",
        "Fidelity is always higher than the witness",
    ],
    correct=1,
    section="6. Fidelity",
    bloom="analyze",
    explanation=(
        "Fidelity captures total overlap with the ideal state. "
        "The witness specifically tests the T-state signature. "
        "A state can have moderate fidelity but low witness if the noise corrupts the magic structure."
    ))

> **Key Insight:** The witness ($) and fidelity ($) measure different things. Fidelity captures *all* deviations from ideal. The witness specifically tests the "magic" property. Both are useful: fidelity is more complete, the witness is more targeted.

---
## 7. The Scoring Formula

The autoresearch harness combines quality, acceptance rate, and cost into a single **score**:

9951	ext{score} = rac{	ext{quality} 	imes 	ext{acceptance\_rate}}{	ext{cost}}9951

where:

9951	ext{quality} = rac{\sum_i w_i \cdot q_i}{\sum_i w_i}9951

with quality components  \in$ {ideal_fidelity, noisy_fidelity, logical_witness, codespace_rate, stability_score, spectator_alignment}, each weighted by $, and:

9951	ext{cost} = 	ext{base} + w_{	ext{2q}} \cdot n_{	ext{2q}} + w_d \cdot d + w_s \cdot s + w_r \cdot r + w_q \cdot q_{	ext{proxy}}9951

In [ ]:
# Build metrics by hand
acceptance_rate = results["acceptance"]["acceptance_rate"]

metrics = EvaluationMetrics(
    ideal_encoded_fidelity=ideal_fidelity,
    noisy_encoded_fidelity=noisy_fidelity,
    logical_magic_witness=lib_witness,
    acceptance_rate=acceptance_rate,
    codespace_rate=np.mean([results[k]["acceptance_rate"] for k in context_names]),
    spectator_logical_z=sz,
    logical_x=lx,
    logical_y=ly,
    two_qubit_count=sum(count_two_qubit_gates(c) for c in transpiled_circuits),
    depth=max(c.depth() for c in transpiled_circuits),
    shot_count=512 * 1 * 4,
)

# Define weights (from rung1.yaml cheap_quality)
score_config = ScoreConfig(
    name="weighted_acceptance_cost",
    cheap_quality=QualityWeights(
        ideal_fidelity=0.10, noisy_fidelity=0.40, logical_witness=0.25,
        codespace_rate=0.15, stability_score=0.05, spectator_alignment=0.05,
    ),
    cost_weights=CostWeights(
        two_qubit_count=0.08, depth=0.01, shot_count=0.00020,
        runtime_estimate=0.015, queue_cost_proxy=0.30,
    ),
    base_cost=1.0,
)

score, quality, cost = score_metrics(metrics, "cheap", score_config)
print(f"Quality:   {quality:.4f}")
print(f"Acceptance:{acceptance_rate:.4f}")
print(f"Cost:      {cost:.4f}")
print(f"Score:     {score:.6f}")
print(f"\nFormula check: {quality:.4f} * {acceptance_rate:.4f} / {cost:.4f} = {quality * acceptance_rate / cost:.6f}")

### The scoring formula

$$\text{score} = \frac{\text{quality} \times \text{acceptance\_rate}}{\text{cost}}$$

In [ ]:
predict_choice(tracker, "q7_score_tension",
    question="If you add stricter verification, what happens to the score?",
    options=[
        "Score always increases \u2014 more checks = better quality",
        "Score always decreases \u2014 more checks = lower acceptance",
        "It depends \u2014 quality improves but acceptance drops; the net effect depends on noise",
    ],
    correct=2,
    section="7. Scoring",
    bloom="evaluate",
    explanation=(
        "Stricter verification filters more errors (higher quality) but rejects more shots (lower acceptance). "
        "At low noise, quality gain dominates. At high noise, acceptance crashes."
    ))
checkpoint_summary(tracker, "7. Scoring")

> **Key Insight:** The score balances three tensions: (1) Higher quality is better, but requires expensive circuits. (2) Strict postselection improves quality but lowers acceptance rate. (3) More shots improve statistics but increase cost. The weights in the config determine the tradeoff.

---
## 8. Parameter Sweep: Optimization Level

Let us see how  affects the score. Higher levels produce more compact circuits but take longer to transpile.

In [ ]:
from autoresearch_quantum.config import load_rung_config

rung_config = load_rung_config("../../configs/rungs/rung1.yaml")
opt_levels = [1, 2, 3]
sweep_results = {}

for opt in opt_levels:
    s = ExperimentSpec(
        rung=1, seed_style="h_p", encoder_style="cx_chain",
        verification="both", postselection="all_measured",
        target_backend="fake_brisbane", noise_backend="fake_brisbane",
        optimization_level=opt, shots=256, repeats=1,
    )
    executor = LocalCheapExecutor()
    tier_result = executor.evaluate(s, rung_config)
    sweep_results[opt] = tier_result
    m = tier_result.metrics
    print(f"opt_level={opt}: score={tier_result.score:.4f}  2q_gates={m.two_qubit_count}  depth={m.depth}  accept={m.acceptance_rate:.3f}  witness={m.logical_magic_witness:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

labels = [f"Level {o}" for o in opt_levels]
scores = [sweep_results[o].score for o in opt_levels]
gates  = [sweep_results[o].metrics.two_qubit_count for o in opt_levels]
depths = [sweep_results[o].metrics.depth for o in opt_levels]
accept = [sweep_results[o].metrics.acceptance_rate for o in opt_levels]
witness_vals = [sweep_results[o].metrics.logical_magic_witness for o in opt_levels]

axes[0].bar(labels, scores, color=["#3498db", "#2ecc71", "#e67e22"])
axes[0].set_title("Score")
axes[0].set_ylabel("Score")

x = np.arange(len(labels))
w = 0.35
axes[1].bar(x - w/2, gates, w, label="2Q Gates", color="#e74c3c")
axes[1].bar(x + w/2, depths, w, label="Depth", color="#9b59b6")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_title("Circuit Cost")
axes[1].legend()

axes[2].bar(x - w/2, accept, w, label="Acceptance", color="#2ecc71")
axes[2].bar(x + w/2, witness_vals, w, label="Witness", color="#3498db")
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels)
axes[2].set_title("Quality Metrics")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
reflect(tracker, "q8_sweep_insight",
    question="Looking at the parameter sweep charts, which optimization level gives the best score and why?",
    section="8. Parameter sweep",
    bloom="evaluate",
    model_answer=(
        "The best optimization level balances gate count reduction against qubit routing overhead. "
        "Level 2 or 3 often wins because aggressive optimization reduces noisy 2-qubit gates. "
        "But the best choice depends on the specific backend topology."
    ))

> **Observe:** Higher optimization levels typically reduce gate count and depth, which lowers cost. But the effect on quality depends on how the transpiler remaps the circuit — sometimes a different layout performs better or worse under that backend's specific noise profile.

---
## 9. Dominant Failure Modes

The harness classifies each experiment into a **dominant failure mode** — the biggest weakness:

| Failure Mode | Trigger | Meaning |
|---|---|---|
| postselection collapse | acceptance_rate < 0.45 | Too many shots fail syndrome checks |
| logical witness erosion | witness < 0.65 | Magic property is severely degraded |
| noise sensitivity | stability_score < 0.75 | Results vary wildly between repeats |
| transpile cost explosion | 2q_gates > 60 or depth > 120 | Circuit too expensive for the quality gained |

In [ ]:
for opt, result in sweep_results.items():
    print(f"opt_level={opt}: failure_mode = {result.metrics.dominant_failure_mode}")

In [ ]:
order(tracker, "q9_failure_ordering",
    instruction="Rank failure modes from least to most severe for magic state quality:",
    items=["high_cost_low_throughput", "poor_acceptance_rate", "low_magic_witness"],
    correct_order=["high_cost_low_throughput", "poor_acceptance_rate", "low_magic_witness"],
    section="9. Failure modes",
    bloom="analyze",
    explanation=(
        "High cost is fixable (fewer gates). Poor acceptance is concerning (too many errors). "
        "Low magic witness is worst \u2014 the state has lost its T-state character."
    ))
checkpoint_summary(tracker, "9. Failure modes")

---
## 10. Factory Throughput Score

The default scorer optimizes . An alternative — **factory throughput** — optimizes for producing accepted magic states per unit cost, penalizing circuit resources more heavily:

9951	ext{throughput} = rac{	ext{acceptance\_rate} 	imes W}{	ext{cost}}9951

This is useful when you care about *yield* in a magic state distillation factory.

In [ ]:
factory_config = ScoreConfig(
    name="factory_throughput",
    cheap_quality=QualityWeights(
        noisy_fidelity=0.3, logical_witness=0.4,
        codespace_rate=0.2, stability_score=0.1,
    ),
)

print(f"{'Metric':<25} {'Weighted Acceptance Cost':>25} {'Factory Throughput':>20}")
print("=" * 72)
for opt in opt_levels:
    m = sweep_results[opt].metrics
    # Reset extra dict for factory calc
    m.extra = {}
    s_wac, q_wac, c_wac = weighted_acceptance_cost(m, "cheap", score_config)
    s_ft, q_ft, c_ft = factory_throughput_score(m, "cheap", factory_config)
    print(f"opt_level={opt:<15} {s_wac:>25.4f} {s_ft:>20.4f}")

In [ ]:
quiz(tracker, "q10_factory_vs_wac",
    question="When would factory throughput scoring beat default WAC scoring?",
    options=[
        "When raw quality matters most",
        "When producing many T-states in a pipeline and throughput matters more than per-state quality",
        "When running on hardware instead of a simulator",
    ],
    correct=1,
    section="10. Factory throughput",
    bloom="evaluate",
    explanation=(
        "Factory throughput penalizes cost more heavily because in a pipeline, "
        "the rate of producing usable T-states matters more than any individual one."
    ))
checkpoint_summary(tracker, "10. Factory throughput")

> **Key Insight:** Different scoring functions rank experiments differently. The choice depends on your goal: maximizing state quality vs maximizing factory throughput. The ratchet is agnostic — it optimizes whichever score you configure.

---
## 11. Summary

You now understand every number the harness produces:

| Concept | What it measures |
|---|---|
| **Acceptance rate** | Fraction of shots surviving postselection |
| **Logical witness** | Certification of magic property ( = 1.0$ for perfect T-state) |
| **Noisy fidelity** | Overlap with ideal encoded state under noise |
| **Score** | quality × acceptance / cost — the single number to optimize |
| **Failure mode** | Biggest weakness of the current configuration |

**Next:** Notebook 3 shows how the ratchet *automatically* explores parameter space to maximize this score.

---
## Final Assessment

In [ ]:
tracker.dashboard()
path = tracker.save()
print(f"\nProgress saved to: {path}")